# 📄 Document Summarization Pipeline
### HTML Cleaning → Preprocessing → Extractive & Abstractive Summarization

This notebook:
1. Accepts any `.txt` file (including HTML content)
2. Strips all HTML tags and attributes
3. Preprocesses text for summarization
4. Runs **Extractive** summarization (TF-IDF + TextRank)
5. Runs **Abstractive** summarization (BART Transformer)
6. Extracts keywords and key statistics
7. Produces visualizations

---
## 0. Install Dependencies

In [2]:
#!pip install nltk beautifulsoup4 sumy transformers torch sentencepiece \
             #wordcloud matplotlib seaborn pandas numpy scikit-learn -q

## 1. Imports

In [ ]:
import re
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from bs4 import BeautifulSoup
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

warnings.filterwarnings('ignore')
for r in ['punkt', 'stopwords', 'wordnet', 'punkt_tab', 'averaged_perceptron_tagger']:
    nltk.download(r, quiet=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

print('All imports complete.')

---
## 2. Load Your Text File
> **Change `FILE_PATH` below.** Use a raw string `r'...'` for Windows paths.

In [ ]:
# ──────────────────────────────────────────────────────────
FILE_PATH     = r'C:\Users\dell\Downloads\Infosys internship\final\nlp_preprocessing_2.txt'  # CHANGE THIS
LANGUAGE      = 'english'
SUMMARY_RATIO = 0.3          # 30% of sentences extracted
# ──────────────────────────────────────────────────────────

with open(FILE_PATH, 'r', encoding='utf-8', errors='replace') as f:
    raw_text = f.read()

print('Loaded: ' + FILE_PATH)
print('Characters : ' + str(len(raw_text)))
print('Lines      : ' + str(raw_text.count(chr(10))))
print()
print('--- Raw Preview (first 600 chars) ---')
print(raw_text[:600])

Loaded: C:\Users\dell\Downloads\Infosys internship\final\nlp_preprocessing_2.txt
Characters : 5473
Lines      : 101

--- Raw Preview (first 600 chars) ---
<html><head><title>AI, Ethics, and Digital Forensics</title></head>
<body>
<div class="content">
<p>Artificial intelligence (AI) has redefined the boundaries of digital forensics — it’s no longer about 
simply analyzing evidence but about understanding <b>patterns</b>, behaviors, and system fingerprints. 
However, data collected from the web often contains unwanted characters &amp; strange symbols like “Â”, “Ã”, or “€” 
due to encoding mismatches. Researchers frequently spend hours cleaning such data before model training.</p>

<p>The TraceFinder project, developed at Infosys Labs, introduces 


---
## 3. Preprocessing Pipeline

> Summarization preprocessing is intentionally **lighter** than sentiment preprocessing.  
> We preserve sentence structure and punctuation — stripping those would break sentence boundaries.

In [ ]:
def remove_html(text):
    """Remove all HTML tags, attributes, scripts and styles."""
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style', 'head', 'meta', 'link', 'noscript']):
        tag.decompose()
    clean = soup.get_text(separator=' ')
    return re.sub(r'\s+', ' ', clean).strip()

def light_clean_for_summary(text):
    """Light clean — preserve sentence structure for summarization."""
    text = re.sub(r'http\S+|www\.\S+', '', text)       # Remove URLs
    text = re.sub(r'\S+@\S+\.\S+', '', text)           # Remove emails
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)        # Remove non-ASCII
    text = re.sub(r'[ \t]+', ' ', text)                 # Normalize spaces
    text = re.sub(r'\n{3,}', '\n\n', text)             # Collapse blank lines
    return text.strip()

def deep_clean_tokens(text):
    """Deep clean for keyword/word-frequency analysis only."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    return [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t.isalpha() and t not in STOP_WORDS and len(t) > 2
    ]

html_removed   = remove_html(raw_text)
clean_text     = light_clean_for_summary(html_removed)
keyword_tokens = deep_clean_tokens(clean_text)

sentences  = sent_tokenize(clean_text)
words_all  = word_tokenize(clean_text)

char_count   = len(clean_text)
word_count   = len(words_all)
sent_count   = len(sentences)
unique_words = len(set(w.lower() for w in words_all if w.isalpha()))
avg_sent_len = word_count / sent_count if sent_count else 0
read_time    = max(1, round(word_count / 200))

LINE = '-' * 50
print('PREPROCESSING COMPLETE')
print(LINE)
print('Characters       : ' + str(char_count))
print('Words            : ' + str(word_count))
print('Unique words     : ' + str(unique_words))
print('Sentences        : ' + str(sent_count))
print('Avg sent. length : ' + str(round(avg_sent_len, 1)) + ' words')
print('Est. read time   : ~' + str(read_time) + ' min')
print('Keyword tokens   : ' + str(len(keyword_tokens)))
print()
print('--- Clean Text Preview ---')
print(clean_text[:500])

PREPROCESSING COMPLETE
--------------------------------------------------
Characters       : 4824
Words            : 824
Unique words     : 375
Sentences        : 45
Avg sent. length : 18.3 words
Est. read time   : ~4 min
Keyword tokens   : 471

--- Clean Text Preview ---
Artificial intelligence (AI) has redefined the boundaries of digital forensics it s no longer about simply analyzing evidence but about understanding patterns , behaviors, and system fingerprints. However, data collected from the web often contains unwanted characters & strange symbols like , , or due to encoding mismatches. Researchers frequently spend hours cleaning such data before model training. The TraceFinder project, developed at Infosys Labs, introduces an innovative method of scanner i


---
## 4. Keyword Extraction (TF-IDF)

In [ ]:
tfidf = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 2))
tfidf_matrix  = tfidf.fit_transform(sentences)
feature_names = tfidf.get_feature_names_out()

avg_tfidf = tfidf_matrix.mean(axis=0).A1
kw_df = pd.DataFrame({'keyword': feature_names, 'score': avg_tfidf})
kw_df = kw_df.sort_values('score', ascending=False).head(20).reset_index(drop=True)

print('TOP 20 KEYWORDS (TF-IDF)')
print('-' * 50)
for idx, row in kw_df.iterrows():
    bar = 'X' * int(row['score'] * 200)
    print(str(idx + 1).rjust(3) + '. ' + row['keyword'].ljust(25) + str(round(row['score'], 4)) + '  ' + bar)

TOP 20 KEYWORDS (TF-IDF)
--------------------------------------------------
  1. data                     0.0945  XXXXXXXXXXXXXXXXXX
  2. model                    0.0786  XXXXXXXXXXXXXXX
  3. like                     0.0722  XXXXXXXXXXXXXX
  4. cleaning                 0.064  XXXXXXXXXXXX
  5. text                     0.0613  XXXXXXXXXXXX
  6. document                 0.0492  XXXXXXXXX
  7. scanner                  0.0487  XXXXXXXXX
  8. scraped                  0.0474  XXXXXXXXX
  9. tampered                 0.0466  XXXXXXXXX
 10. evidence                 0.0456  XXXXXXXXX
 11. lda                      0.0406  XXXXXXXX
 12. proper                   0.0385  XXXXXXX
 13. preprocessing            0.038  XXXXXXX
 14. div                      0.0373  XXXXXXX
 15. similar                  0.0371  XXXXXXX
 16. forensic                 0.0354  XXXXXXX
 17. forensics                0.0353  XXXXXXX
 18. example                  0.035  XXXXXX
 19. tracefinder              0.0337  XXXXXX
 20. noi

---
## 5. Extractive Summarization (TextRank + LSA + LexRank)

In [ ]:
N_SENTENCES = max(3, int(sent_count * SUMMARY_RATIO))
SEP = '=' * 60

print('Extracting ' + str(N_SENTENCES) + ' sentences from ' + str(sent_count) + ' total')
print()

parser  = PlaintextParser.from_string(clean_text, Tokenizer(LANGUAGE))
stemmer = Stemmer(LANGUAGE)

SUMMARIZERS = {
    'TextRank': TextRankSummarizer(stemmer),
    'LSA':      LsaSummarizer(stemmer),
    'LexRank':  LexRankSummarizer(stemmer),
}

summaries = {}
for name, summarizer in SUMMARIZERS.items():
    summarizer.stop_words = get_stop_words(LANGUAGE)
    extracted = summarizer(parser.document, N_SENTENCES)
    summaries[name] = [str(s) for s in extracted]

for name, sents in summaries.items():
    header = name + ' SUMMARY (' + str(len(sents)) + ' sentences)'
    print(SEP)
    print('  ' + header)
    print(SEP)
    for i, sent in enumerate(sents, 1):
        print('  [' + str(i) + '] ' + sent)
    print()

Extracting 13 sentences from 45 total

  TextRank SUMMARY (13 sentences)
  [1] Researchers frequently spend hours cleaning such data before model training.
  [2] Using convolutional neural networks (CNNs) with handcrafted statistical features like LBP (Local Binary Pattern) and FFT (Fast Fourier Transform) energy distributions, the model can trace the origin of a document scan.
  [3] Common Web Scraping Issues HTML elements like <div>, <meta>, or <script> appear inside scraped text.
  [4] After cleaning, topic modeling techniques like Latent Dirichlet Allocation (LDA) can discover hidden themes in the data.
  [5] Texture and frequency-based features During early experiments, LDA showed promise in grouping similar documents together, though the quality of topics depended heavily on preprocessing.
  [6] Dataset Construction To train AI models for forensic scanner identification, researchers compiled thousands of authentic and tampered document images.
  [7] However, in some scraped text,

### Consensus Summary
Sentences chosen by **2+ out of 3** summarizers — the most reliable extractive result.

In [ ]:
all_summary_sents = [s for v in summaries.values() for s in v]
vote_counts = Counter(all_summary_sents)

consensus = [s for s, count in vote_counts.items() if count >= 2]

def sentence_position(sent):
    for i, orig in enumerate(sentences):
        if sent.strip() in orig or orig.strip() in sent:
            return i
    return 9999

consensus_ordered = sorted(consensus, key=sentence_position)

if not consensus_ordered:
    consensus_ordered = summaries['TextRank']
    print('No consensus found — falling back to TextRank.')

extractive_text = ' '.join(consensus_ordered)
SEP = '=' * 62

print(SEP)
print('  CONSENSUS EXTRACTIVE SUMMARY (Best sentences)')
print(SEP)
for i, sent in enumerate(consensus_ordered, 1):
    print()
    print('  [' + str(i) + '] ' + sent)
print()
ext_word_count = len(extractive_text.split())
print('  Summary : ' + str(len(consensus_ordered)) + ' sentences | ~' +
      str(ext_word_count) + ' words | ' +
      str(round(ext_word_count / word_count * 100)) + '% of original')

  CONSENSUS EXTRACTIVE SUMMARY (Best sentences)

  [1] The TraceFinder project, developed at Infosys Labs, introduces an innovative method of scanner identification.

  [2] Using convolutional neural networks (CNNs) with handcrafted statistical features like LBP (Local Binary Pattern) and FFT (Fast Fourier Transform) energy distributions, the model can trace the origin of a document scan.

  [3] Common Web Scraping Issues HTML elements like <div>, <meta>, or <script> appear inside scraped text.

  [4] After cleaning, topic modeling techniques like Latent Dirichlet Allocation (LDA) can discover hidden themes in the data.

  [5] Texture and frequency-based features During early experiments, LDA showed promise in grouping similar documents together, though the quality of topics depended heavily on preprocessing.

  [6] Dataset Construction To train AI models for forensic scanner identification, researchers compiled thousands of authentic and tampered document images.

  [7] However, in so

---
## 6. Abstractive Summarization (BART)
> First run downloads ~1.6 GB. Subsequent runs are instant (cached).

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

print('Loading BART model and tokenizer (first run downloads ~1.6 GB)...')
MODEL_NAME = 'facebook/bart-large-cnn'
tokenizer  = BartTokenizer.from_pretrained(MODEL_NAME)
model      = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
print('Model loaded.')

MAX_CHARS = 3500

def chunk_text(text, max_chars=MAX_CHARS):
    sents   = sent_tokenize(text)
    chunks  = []
    current = ''
    for s in sents:
        if len(current) + len(s) < max_chars:
            current += ' ' + s
        else:
            if current:
                chunks.append(current.strip())
            current = s
    if current:
        chunks.append(current.strip())
    return chunks

def bart_summarize(text, max_length=150, min_length=40):
    inputs = tokenizer(
        text,
        max_length=1024,
        return_tensors='pt',
        truncation=True
    )
    summary_ids = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Process each chunk
chunks = chunk_text(clean_text)
print('Document split into ' + str(len(chunks)) + ' chunk(s) for BART.')

chunk_summaries = []
for i, chunk in enumerate(chunks, 1):
    print('  Summarizing chunk ' + str(i) + '/' + str(len(chunks)) + '...')
    chunk_summaries.append(bart_summarize(chunk))

# If multiple chunks, do a second-pass summary on the combined chunk summaries
if len(chunk_summaries) > 1:
    combined = ' '.join(chunk_summaries)
    abstractive_summary = bart_summarize(combined[:MAX_CHARS], max_length=200, min_length=60)
else:
    abstractive_summary = chunk_summaries[0]

SEP = '=' * 62
print()
print(SEP)
print('  ABSTRACTIVE SUMMARY (BART)')
print(SEP)
print(abstractive_summary)
print()
print('  Words: ' + str(len(abstractive_summary.split())))


Loading BART model and tokenizer (first run downloads ~1.6 GB)...


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

---
## 7. Final Detailed Report

In [ ]:
freq = FreqDist(keyword_tokens)
SEP  = '=' * 62

compression_pct = round(len(extractive_text.split()) / word_count * 100) if word_count else 0

print(SEP)
print('          FINAL DOCUMENT SUMMARY REPORT')
print(SEP)
print('  File            : ' + FILE_PATH)
print('  Characters      : ' + str(char_count))
print('  Words           : ' + str(word_count))
print('  Sentences       : ' + str(sent_count))
print('  Unique words    : ' + str(unique_words))
print('  Avg sent. len   : ' + str(round(avg_sent_len, 1)) + ' words')
print('  Est. read time  : ~' + str(read_time) + ' min')
print()
print('  Extractive summary    : ' + str(len(consensus_ordered)) + ' sentences | ' +
      str(len(extractive_text.split())) + ' words | ' + str(compression_pct) + '% of original')
print('  Abstractive summary   : ' + str(len(abstractive_summary.split())) + ' words')
print()
print('  TOP KEYWORDS')
print('  ' + '-' * 45)
for i, (w, c) in enumerate(freq.most_common(15), 1):
    print('  ' + str(i).rjust(2) + '. ' + w.ljust(22) + '(' + str(c) + ' occurrences)')
print()
print('  EXTRACTIVE SUMMARY')
print('  ' + '-' * 45)
for i, s in enumerate(consensus_ordered, 1):
    print('  [' + str(i) + '] ' + s)
print()
print('  ABSTRACTIVE SUMMARY (BART)')
print('  ' + '-' * 45)
print('  ' + abstractive_summary)
print()
print(SEP)

---
## 8. Visualizations

In [ ]:
fig = plt.figure(figsize=(20, 20))
fig.suptitle('Document Summary Dashboard', fontsize=22, fontweight='bold', y=1.01)

# ── 1: Document Statistics ────────────────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
stats  = {'Words': word_count, 'Sentences': sent_count,
          'Unique\nWords': unique_words, 'Keywords': len(keyword_tokens)}
bars = ax1.bar(stats.keys(), stats.values(),
               color=['#3498db','#2ecc71','#e67e22','#9b59b6'], edgecolor='white', linewidth=1.5)
ax1.bar_label(bars, fmt='{:,.0f}', fontsize=9, fontweight='bold')
ax1.set_title('Document Statistics', fontweight='bold')
ax1.set_ylabel('Count')

# ── 2: Compression Ratio ─────────────────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
labels = ['Original\nDocument', 'Extractive\nSummary', 'Abstractive\nSummary']
sizes  = [word_count, len(extractive_text.split()), len(abstractive_summary.split())]
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars2  = ax2.barh(labels, sizes, color=colors, edgecolor='white', linewidth=1.5)
ax2.bar_label(bars2, labels=[str(s) + ' words' for s in sizes], padding=4, fontsize=9)
ax2.set_title('Words: Original vs Summaries', fontweight='bold')
ax2.set_xlabel('Word Count')

# ── 3: Top 15 Keywords (TF-IDF) ───────────────────────────────────
ax3 = fig.add_subplot(3, 3, 3)
top15   = kw_df.head(15)
palette = sns.color_palette('Blues_r', len(top15))
ax3.barh(top15['keyword'][::-1], top15['score'][::-1], color=palette)
ax3.set_title('Top 15 Keywords (TF-IDF)', fontweight='bold')
ax3.set_xlabel('Average TF-IDF Score')

# ── 4: Sentence Length Distribution ──────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
sent_lengths = [len(s.split()) for s in sentences]
ax4.hist(sent_lengths, bins=20, color='#3498db', edgecolor='white', linewidth=0.8, alpha=0.85)
ax4.axvline(np.mean(sent_lengths), color='#e74c3c', linestyle='--', linewidth=2,
            label='Mean: ' + str(round(np.mean(sent_lengths), 1)))
ax4.set_title('Sentence Length Distribution', fontweight='bold')
ax4.set_xlabel('Words per Sentence')
ax4.set_ylabel('Count')
ax4.legend(fontsize=9)

# ── 5: Summarizer Agreement ───────────────────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
vote_data = {
    'Only 1\nSummarizer':  sum(1 for v in vote_counts.values() if v == 1),
    'In 2\nSummarizers':   sum(1 for v in vote_counts.values() if v == 2),
    'All 3\nSummarizers':  sum(1 for v in vote_counts.values() if v == 3),
}
bars5 = ax5.bar(vote_data.keys(), vote_data.values(),
                color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='white', linewidth=1.5)
ax5.bar_label(bars5, fmt='{:.0f}', fontsize=10, fontweight='bold')
ax5.set_title('Summarizer Agreement\n(sentences chosen by each count)', fontweight='bold', fontsize=10)
ax5.set_ylabel('Sentences')

# ── 6: Word Frequency Top 20 ─────────────────────────────────────
ax6 = fig.add_subplot(3, 3, 6)
top20 = freq.most_common(20)
if top20:
    tw, tc = zip(*top20)
    ax6.bar(tw, tc, color=sns.color_palette('husl', 20), edgecolor='white')
    ax6.set_title('Top 20 Word Frequencies', fontweight='bold')
    ax6.set_xlabel('Word')
    ax6.set_ylabel('Count')
    plt.setp(ax6.get_xticklabels(), rotation=45, ha='right', fontsize=8)

# ── 7: Coverage Pie ───────────────────────────────────────────────
ax7 = fig.add_subplot(3, 3, 7)
ext_w = len(extractive_text.split())
abs_w = len(abstractive_summary.split())
rest  = max(0, word_count - ext_w)
ax7.pie(
    [ext_w, abs_w, rest],
    labels=['Extractive\nSummary', 'Abstractive\nSummary', 'Rest of Doc'],
    colors=['#f39c12', '#2ecc71', '#ecf0f1'],
    autopct='%1.0f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
ax7.set_title('Summary Coverage vs Full Document', fontweight='bold')

# ── 8: Word Cloud — Full Document ────────────────────────────────
ax8 = fig.add_subplot(3, 3, 8)
wc_text = ' '.join(keyword_tokens)
if wc_text:
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap='Blues', max_words=100, collocations=False).generate(wc_text)
    ax8.imshow(wc, interpolation='bilinear')
ax8.axis('off')
ax8.set_title('Document Word Cloud', fontweight='bold')

# ── 9: Word Cloud — Extractive Summary ───────────────────────────
ax9 = fig.add_subplot(3, 3, 9)
summary_tokens = deep_clean_tokens(extractive_text)
wc_sum_text    = ' '.join(summary_tokens)
if wc_sum_text:
    wc2 = WordCloud(width=600, height=400, background_color='white',
                    colormap='Oranges', max_words=80, collocations=False).generate(wc_sum_text)
    ax9.imshow(wc2, interpolation='bilinear')
ax9.axis('off')
ax9.set_title('Summary Word Cloud', fontweight='bold')

plt.tight_layout()
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved as summary_dashboard.png')

---
## 9. Export Results

In [ ]:
output_txt = 'document_summary.txt'
with open(output_txt, 'w', encoding='utf-8') as f:
    f.write('DOCUMENT SUMMARY REPORT\n')
    f.write('=' * 60 + '\n')
    f.write('Source File : ' + FILE_PATH + '\n')
    f.write('Words       : ' + str(word_count) + '\n')
    f.write('Sentences   : ' + str(sent_count) + '\n')
    f.write('Est. Read   : ~' + str(read_time) + ' min\n')
    f.write('\n')
    f.write('TOP KEYWORDS\n' + '-' * 40 + '\n')
    for i, (w, c) in enumerate(freq.most_common(15), 1):
        f.write(str(i).rjust(2) + '. ' + w + ' (' + str(c) + ')\n')
    f.write('\n')
    f.write('EXTRACTIVE SUMMARY (Consensus)\n' + '-' * 40 + '\n')
    for i, s in enumerate(consensus_ordered, 1):
        f.write('[' + str(i) + '] ' + s + '\n')
    f.write('\n')
    f.write('ABSTRACTIVE SUMMARY (BART)\n' + '-' * 40 + '\n')
    f.write(abstractive_summary + '\n')

results_df = pd.DataFrame({
    'Type':     ['Extractive'] * len(consensus_ordered) + ['Abstractive'],
    'Sentence': consensus_ordered + [abstractive_summary]
})
results_df.to_csv('summary_results.csv', index=False)

print('Summary saved to "' + output_txt + '"')
print('Structured results saved to "summary_results.csv"')
display(results_df)